# Семинар: Свёрточные нейросети для компьютерного зрения

В этом семинаре мы научимся строить и обучать свёрточные нейросети (CNN) для классификации изображений.

**План:**
1. Датасет CIFAR-10
2. Базовая полносвязная сеть (baseline)
3. Свёрточная нейросеть
4. Batch Normalization
5. Аугментация данных

## Датасет CIFAR-10

- 60 000 цветных изображений 32x32 (3 канала)
- 10 классов: самолёт, автомобиль, птица, кошка, олень, собака, лягушка, лошадь, корабль, грузовик
- Стандартный бенчмарк для оценки CNN

<img src="https://github.com/yandexdataschool/deep_vision_and_graphics/blob/fall22/week02-convnets/cifar10.jpg?raw=1" style="width:80%">

In [ ]:
# Если запускаете в Colab, раскомментируйте:
# !wget https://raw.githubusercontent.com/yandexdataschool/Practical_DL/refs/heads/fall25/week03_convnets/cifar.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cifar import load_cifar10
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10("cifar_data")

class_names = np.array(['самолёт', 'автомобиль', 'птица', 'кошка', 'олень',
                        'собака', 'лягушка', 'лошадь', 'корабль', 'грузовик'])

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# Визуализация примеров
plt.figure(figsize=[12, 10])
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.xlabel(class_names[y_train[i]])
    plt.imshow(np.transpose(X_train[i], [1, 2, 0]))
plt.tight_layout()
plt.show()

---
## 1. Baseline: полносвязная сеть

Начнём с простой полносвязной сети. Она будет нашим baseline, который мы затем улучшим с помощью свёрток.

Обратите внимание:
- `nn.Flatten()` превращает изображение 3x32x32 в вектор длины 3072
- Последний слой выдаёт **логиты** (без softmax) — это стандарт для PyTorch, `F.cross_entropy` сам применяет softmax

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3 * 32 * 32, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

In [ ]:
def compute_loss(X_batch, y_batch):
    """Вычисление кросс-энтропии для батча."""
    X_batch = torch.as_tensor(X_batch, dtype=torch.float32)
    y_batch = torch.as_tensor(y_batch, dtype=torch.int64)
    logits = model(X_batch)
    return F.cross_entropy(logits, y_batch).mean()

# Проверка
compute_loss(X_train[:5], y_train[:5])

### Обучение на мини-батчах

40 000 изображений — слишком много для полного batch SGD. Будем обучать на мини-батчах.

In [ ]:
def iterate_minibatches(X, y, batchsize):
    """Генератор мини-батчей."""
    indices = np.random.permutation(np.arange(len(X)))
    for start in range(0, len(indices), batchsize):
        ix = indices[start: start + batchsize]
        yield X[ix], y[ix]

In [ ]:
import time

opt = torch.optim.SGD(model.parameters(), lr=0.01)

train_loss = []
val_accuracy = []
num_epochs = 15  # для baseline достаточно ~15 эпох
batch_size = 50

for epoch in range(num_epochs):
    start_time = time.time()

    # Обучение
    model.train(True)
    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        loss = compute_loss(X_batch, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        train_loss.append(loss.item())

    # Валидация
    model.train(False)
    with torch.no_grad():
        for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size):
            logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
            y_pred = logits.argmax(-1).detach().numpy()
            val_accuracy.append(np.mean(y_batch == y_pred))

    print(f"Эпоха {epoch+1}/{num_epochs} ({time.time()-start_time:.1f}с)")
    print(f"  Train loss: {np.mean(train_loss[-len(X_train)//batch_size:]):.4f}")
    print(f"  Val accuracy: {np.mean(val_accuracy[-len(X_val)//batch_size:])*100:.1f}%")

In [ ]:
# Тест baseline
model.train(False)
test_batch_acc = []
for X_batch, y_batch in iterate_minibatches(X_test, y_test, 500):
    logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
    y_pred = logits.max(1)[1].data.numpy()
    test_batch_acc.append(np.mean(y_batch == y_pred))

print(f"Baseline test accuracy: {np.mean(test_batch_acc)*100:.1f}%")

---
## Задание 1: Небольшая свёрточная сеть

Создайте свёрточную нейросеть с такой архитектурой:
1. Свёртка 3x3 с 10 фильтрами + ReLU
2. MaxPooling 2x2
3. Flatten
4. Полносвязный слой на 100 нейронов + ReLU
5. Dropout 10%
6. Выходной слой (10 классов)

**Подсказка по размерам:**
- Вход: `[batch, 3, 32, 32]`
- После Conv2d(3, 10, 3): `[batch, 10, 30, 30]`
- После MaxPool2d(2): `[batch, 10, 15, 15]`
- После Flatten: `[batch, 10*15*15]` = `[batch, 2250]`

**Если не хотите считать размеры вручную**, запустите `compute_loss` — ошибка покажет правильный размер.

In [ ]:
model = nn.Sequential(
    # YOUR CODE: Conv2d, ReLU, MaxPool2d, Flatten, Linear, ReLU, Dropout, Linear
)

# Проверка
compute_loss(X_train[:5], y_train[:5])

In [ ]:
# Обучение (используйте Adam)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss = []
val_accuracy = []
num_epochs = 20
batch_size = 50

for epoch in range(num_epochs):
    start_time = time.time()

    model.train(True)
    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        loss = compute_loss(X_batch, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        train_loss.append(loss.item())

    model.train(False)
    with torch.no_grad():
        for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size):
            logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
            y_pred = logits.argmax(-1).detach().numpy()
            val_accuracy.append(np.mean(y_batch == y_pred))

    print(f"Эпоха {epoch+1}/{num_epochs} ({time.time()-start_time:.1f}с)")
    print(f"  Train loss: {np.mean(train_loss[-len(X_train)//batch_size:]):.4f}")
    print(f"  Val accuracy: {np.mean(val_accuracy[-len(X_val)//batch_size:])*100:.1f}%")

In [ ]:
# Тест
model.train(False)
test_batch_acc = []
for X_batch, y_batch in iterate_minibatches(X_test, y_test, 500):
    logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
    y_pred = logits.max(1)[1].data.numpy()
    test_batch_acc.append(np.mean(y_batch == y_pred))

test_accuracy = np.mean(test_batch_acc) * 100
print(f"CNN test accuracy: {test_accuracy:.1f}%")
assert test_accuracy > 50, "Accuracy должен быть > 50%. Проверьте архитектуру."

---
## Задание 2: Добавьте Batch Normalization

Batch Normalization нормализует выходы слоя по мини-батчу. Это помогает:
- Ускорить сходимость
- Позволяет использовать более высокий learning rate
- Действует как лёгкая регуляризация

**Где ставить BatchNorm:** обычно после линейного/свёрточного слоя, но **перед** функцией активации.

```python
nn.Conv2d(3, 10, 3),
nn.BatchNorm2d(10),  # <-- после свёртки, перед ReLU
nn.ReLU(),
```

Добавьте BatchNorm в вашу сеть и добейтесь **>60%** на валидации.

In [ ]:
model = nn.Sequential(
    # YOUR CODE: та же архитектура, но с BatchNorm2d и BatchNorm1d
)

# Проверка
compute_loss(X_train[:5], y_train[:5])

In [ ]:
# Обучение с BatchNorm
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss = []
val_accuracy = []
num_epochs = 20
batch_size = 50

for epoch in range(num_epochs):
    start_time = time.time()

    model.train(True)
    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        loss = compute_loss(X_batch, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        train_loss.append(loss.item())

    model.train(False)
    with torch.no_grad():
        for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size):
            logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
            y_pred = logits.argmax(-1).detach().numpy()
            val_accuracy.append(np.mean(y_batch == y_pred))

    print(f"Эпоха {epoch+1}/{num_epochs} ({time.time()-start_time:.1f}с)")
    print(f"  Train loss: {np.mean(train_loss[-len(X_train)//batch_size:]):.4f}")
    print(f"  Val accuracy: {np.mean(val_accuracy[-len(X_val)//batch_size:])*100:.1f}%")

---
## Задание 3: Аугментация данных

Аугментация — это искусственное увеличение обучающей выборки за счёт случайных преобразований:
- Случайные обрезки (RandomCrop)
- Горизонтальное отражение (RandomHorizontalFlip)
- Повороты (RandomRotation)
- Нормализация (Normalize)

В PyTorch для этого используются `torchvision.transforms`.

In [ ]:
from torchvision import transforms
from torchvision.datasets import CIFAR10

# Статистики нормализации для CIFAR-10
means = np.array((0.4914, 0.4822, 0.4465))
stds = np.array((0.2023, 0.1994, 0.2010))

# Пайплайн для обучения (с аугментацией)
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomRotation([-30, 30]),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

# Пайплайн для теста (только нормализация)
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

In [ ]:
train_dataset = CIFAR10("./cifar_data/", train=True, transform=transform_train, download=True)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=1)

# Посмотрим на аугментированные примеры
for (x_batch, y_batch) in train_dataloader:
    print(f'X: {x_batch.shape}, y: {y_batch.shape}')

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        img = x_batch[i].numpy().transpose([1, 2, 0]) * stds + means
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(class_names[y_batch[i]])
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    break  # показываем только один батч

In [ ]:
# Создайте тестовый DataLoader
test_dataset = # YOUR CODE: CIFAR10 с transform_test, train=False
test_dataloader = # YOUR CODE: DataLoader с batch_size=500

### Задание: обучите сеть с аугментацией

Перепишите цикл обучения, используя `train_dataloader` вместо `iterate_minibatches`. Модель может быть любой из предыдущих заданий (или лучше).

In [ ]:
# YOUR CODE: определите модель, оптимизатор и цикл обучения с DataLoader

---
## 4. Конфигурация эксперимента: dataclass

Когда параметров много (learning rate, batch size, число эпох, архитектура...), удобно собрать их в одну структуру.

`dataclass` — стандартный Python-способ создать конфиг без лишнего кода.

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainConfig:
    """Все гиперпараметры эксперимента в одном месте."""
    # Данные
    batch_size: int = 64
    num_workers: int = 2

    # Оптимизация
    learning_rate: float = 1e-3
    num_epochs: int = 30
    optimizer: str = 'adam'          # 'adam', 'sgd'

    # Scheduler
    use_scheduler: bool = True
    scheduler_patience: int = 5     # для ReduceLROnPlateau

    # Early stopping
    early_stopping_patience: int = 10

    # Аугментация
    use_augmentation: bool = True

cfg = TrainConfig()
print(cfg)

Легко менять один параметр:
```python
cfg = TrainConfig(learning_rate=0.01, batch_size=128)
```

Можно сохранить/загрузить:
```python
import json
from dataclasses import asdict
json.dump(asdict(cfg), open('config.json', 'w'))  # сохранить
cfg = TrainConfig(**json.load(open('config.json')))  # загрузить
```

---
## 5. Полный цикл обучения

Соберём вместе всё, что узнали: DataLoader, модель, оптимизатор, scheduler, early stopping, логирование.

### 5.1 DataLoader с аугментацией

In [ ]:
from torchvision import transforms
from torchvision.datasets import CIFAR10

means = (0.4914, 0.4822, 0.4465)
stds = (0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

train_dataset = CIFAR10('./cifar_data/', train=True, transform=transform_train, download=True)
test_dataset = CIFAR10('./cifar_data/', train=False, transform=transform_test, download=True)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=500, shuffle=False, num_workers=cfg.num_workers
)

print(f'Train: {len(train_dataset)}, Test: {len(test_dataset)}')

### 5.2 Модель

Создайте свою модель. Используйте паттерн: `Conv → BN → ReLU → Conv → BN → ReLU → Pool`.

**Задание:** напишите модель с хотя бы 2 свёрточными блоками.

In [ ]:
model = nn.Sequential(
    # YOUR CODE: постройте свёрточную сеть
    # Рекомендуемая структура:
    # Блок 1: Conv(3->32) -> BN -> ReLU -> Conv(32->32) -> BN -> ReLU -> MaxPool -> Dropout2d
    # Блок 2: Conv(32->64) -> BN -> ReLU -> Conv(64->64) -> BN -> ReLU -> MaxPool -> Dropout2d
    # Голова:  Flatten -> Linear -> BN1d -> ReLU -> Dropout -> Linear(10)
)

# Проверка: прогоним один батч
with torch.no_grad():
    dummy = torch.randn(2, 3, 32, 32)
    out = model(dummy)
    print(f'Output shape: {out.shape}')  # должно быть [2, 10]

### 5.3 Оптимизатор и Scheduler

**Scheduler** уменьшает learning rate по ходу обучения. Основные варианты:

| Scheduler | Когда использовать |
|-----------|-------------------|
| `ReduceLROnPlateau` | LR уменьшается, если метрика не улучшается N эпох |
| `CosineAnnealingLR` | LR плавно падает по косинусоиде |
| `OneCycleLR` | Разогрев + спад за один цикл (самый современный) |
| `StepLR` | LR уменьшается каждые N эпох (простой) |

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.learning_rate
)

# Scheduler: уменьшает lr, если val accuracy не растёт
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',              # следим за accuracy (чем больше, тем лучше)
    factor=0.5,              # lr *= 0.5
    patience=cfg.scheduler_patience,
    verbose=True
)

### 5.4 Функции train_epoch и evaluate

In [ ]:
def train_epoch(model, loader, optimizer, device='cpu'):
    """Одна эпоха обучения. Возвращает средний loss."""
    model.train()
    losses = []

    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # YOUR CODE: стандартный цикл
        # 1. logits = model(x_batch)
        # 2. loss = F.cross_entropy(logits, y_batch)
        # 3. loss.backward()
        # 4. optimizer.step()
        # 5. optimizer.zero_grad()

        losses.append(loss.item())

    return np.mean(losses)


@torch.no_grad()
def evaluate(model, loader, device='cpu'):
    """Оценка модели. Возвращает (loss, accuracy)."""
    model.eval()
    losses, correct, total = [], 0, 0

    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # YOUR CODE:
        # 1. logits = model(x_batch)
        # 2. loss = F.cross_entropy(logits, y_batch)
        # 3. preds = logits.argmax(dim=1)
        # 4. correct += (preds == y_batch).sum().item()
        # 5. total += y_batch.size(0)

        losses.append(loss.item())

    return np.mean(losses), correct / total

### 5.5 Задание: полный цикл обучения

Реализуйте цикл с:
- Обучением и валидацией каждую эпоху
- Scheduler (уменьшает lr)
- Early stopping (остановка, если accuracy не растёт)
- Сохранением лучшей модели

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Device: {device}')

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_acc = 0.0
patience_counter = 0

for epoch in range(cfg.num_epochs):
    t0 = time.time()

    # --- Обучение ---
    train_loss = train_epoch(model, train_loader, optimizer, device)

    # --- Валидация ---
    val_loss, val_acc = evaluate(model, test_loader, device)

    # --- Scheduler ---
    current_lr = optimizer.param_groups[0]['lr']
    if cfg.use_scheduler:
        scheduler.step(val_acc)  # передаём метрику для ReduceLROnPlateau

    # --- Логирование ---
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    print(f'Эпоха {epoch+1:02d}/{cfg.num_epochs} ({time.time()-t0:.1f}с) | '
          f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
          f'val_acc={val_acc*100:.1f}% | lr={current_lr:.1e}')

    # --- Сохранение лучшей модели ---
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        patience_counter = 0
        print(f'  ✓ Лучшая модель сохранена (acc={best_acc*100:.1f}%)')
    else:
        patience_counter += 1

    # --- Early stopping ---
    if patience_counter >= cfg.early_stopping_patience:
        print(f'Early stopping на эпохе {epoch+1}')
        break

print(f'\nЛучшая accuracy: {best_acc*100:.1f}%')

In [ ]:
# Визуализация обучения
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].set_title('Loss')

axes[1].plot([a * 100 for a in history['val_acc']])
axes[1].set_xlabel('Эпоха'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Val Accuracy')

axes[2].plot(history['lr'])
axes[2].set_xlabel('Эпоха'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')

plt.tight_layout()
plt.show()

---
## 6. Шпаргалка: что делать, если...

### Модель переобучается (train loss падает, val loss растёт)

| Что делать | Как |
|-----------|-----|
| **Аугментация** | `RandomCrop`, `HorizontalFlip`, `ColorJitter`, `RandomErasing` |
| **Dropout** | `nn.Dropout(0.3)` после Linear, `nn.Dropout2d(0.2)` после Conv |
| **Меньше параметров** | Уменьшить число фильтров или слоёв |
| **Early stopping** | Остановиться, пока val не ухудшилась |
| **Label smoothing** | `F.cross_entropy(logits, y, label_smoothing=0.1)` |

### Модель недообучается (train loss не падает)

| Что делать | Как |
|-----------|-----|
| **Больше параметров** | Добавить фильтры (32→64→128), слои |
| **Убрать регуляризацию** | Убрать Dropout, уменьшить weight_decay |
| **Увеличить lr** | Попробовать lr=3e-3 или lr=1e-2 |
| **BatchNorm** | Ускоряет обучение, стабилизирует градиенты |

### Loss стал NaN / Inf

| Причина | Решение |
|---------|--------|
| Слишком большой lr | Уменьшить в 3-10 раз |
| log(0) | Добавить `+ 1e-8` или использовать `F.cross_entropy` |
| Gradient explosion | Gradient clipping: `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` |

### Хочу выжать максимум

| Техника | Описание |
|---------|----------|
| **CosineAnnealing / OneCycleLR** | Лучшие scheduler'ы для финальной точности |
| **Mixup / CutMix** | Продвинутая аугментация |
| **EMA** | Усреднение весов за обучение |
| **Test-Time Augmentation (TTA)** | Аугментация при инференсе, среднее предсказание |
| **Pretrained backbone** | `torchvision.models.resnet18(pretrained=True)` |

### Чек-лист архитектуры CNN

```
✅ Conv(3x3, padding=1) сохраняет размер — используй padding!
✅ Фильтры растут: 32 → 64 → 128 (чем глубже, тем больше)
✅ BatchNorm ПОСЛЕ Conv/Linear, ПЕРЕД ReLU
✅ Pooling уменьшает размер, Dropout — регуляризация
✅ Последний слой — Linear(N, num_classes), БЕЗ softmax
✅ F.cross_entropy сам применит softmax + log

❌ НЕ ставь Softmax в скрытых слоях
❌ НЕ ставь Dropout после Softmax
❌ НЕ ставь kernel_size > 7 (используй несколько 3x3 вместо одной 7x7)
❌ НЕ забывай model.train() / model.eval()
❌ НЕ забывай optimizer.zero_grad()
```

### Residual Block (skip connection)

Идея ResNet: вместо `y = F(x)` обучаем `y = F(x) + x` — остаточную (residual) связь.
Это помогает градиентам течь через глубокие сети.

```python
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.block(x) + x)  # skip connection
```

---
## Бонус: улучшение модели

Попробуйте улучшить accuracy, используя:
- Больше свёрточных слоёв
- Больше фильтров
- Другие оптимизаторы и learning rate
- Архитектуры ResNet, Inception
- Weight decay (L2 регуляризация)
- Dropout

**Ориентиры:**
- 50% — минимум
- 60% — BatchNorm
- 70% — хорошая архитектура + аугментация
- 80%+ — продвинутая архитектура + тонкая настройка

In [ ]:
# YOUR CODE: экспериментируйте!